# BioMA Land Cover Classification with Random Forest

This workflow will demonstrate how one can prepare a Land Use/Land Cover (LULC) Classification map using PlanetScope Surface Reflectance Basemaps (4.77m).

Featured are sections walking through:
1) Setting up the training dataset
2) Sampling representative points
3) Extracting features from the basemap for each representative point
4) Training a Random Forest classifier.
5) Evaluating performance against the training dataset, and saving the output model.

In [ ]:
#!pip install geopandas rasterio shapely scikit-learn seaborn tqdm joblib

In [ ]:
import os
import json
import warnings
from datetime import datetime
import time

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from rasterio.windows import Window
import seaborn as sns
from shapely.geometry import Point
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import plot_tree
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import joblib

# Configuration
warnings.filterwarnings('ignore')

In [ ]:
# NOTE: This is customized to this AOI, if AOI is changed check that EPSG assigns a projection that operates in meters.
def ops_in_meters(gdf, estimate_area=False, buffer=None, epsg_code=32618):
    orig_crs = gdf.crs
    # Reproject to local UTM Zone for area calculation in meters
    gdf_modified = gdf.to_crs(epsg=epsg_code)
    if buffer is not None:
        if buffer < 1 and buffer > -1:
            print("Warning: buffer should be in meters, but a decimal near 0 was detected.")
        gdf_modified.geometry = gdf_modified.buffer(buffer)

    if estimate_area:
        gdf_modified['area_km2'] = gdf_modified.geometry.area / 1e6  # Convert from m² to km²
        gdf_modified['area_km2'] = gdf_modified['area_km2'].round(1)

    gdf_out = gdf_modified.to_crs(orig_crs) # Restore original CRS

    return gdf_out

In [ ]:
# CONFIGURATION
TRAINING_DATA = os.path.join(
    "BioMA_LULC", "BioMA_LULC_clipped.geojson")
BASEMAP_DIR = os.path.join(
    "basemaps", "4eea1086-5a5e-45bd-b388-87b7b6ddc8be", "ps_monthly_sen2_normalized_analytic_subscription_2025_06_mosaic")
BASEMAP_PATH = os.path.join(
    BASEMAP_DIR, "ps_monthly_sen2_normalized_analytic_subscription_2025_06_mosaic_merge_clip.tif")
UDM2_PATH = os.path.join(
    BASEMAP_DIR, "ps_monthly_sen2_normalized_analytic_subscription_2025_06_mosaic_ortho_udm2_merge_clip.tif")
OUTPUT_DIR = "outputs"
# Dry season: June-September
TOI = "2025-06"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# validate existing files
for ff in [TRAINING_DATA, BASEMAP_PATH, UDM2_PATH]:
    if not os.path.exists(ff):
        raise FileNotFoundError(f"Required file not found: {ff}")

print("Configuration:")
print(f"  Training data: {TRAINING_DATA}")
print(f"  Basemap: {BASEMAP_PATH}")
print(f"  Output: {OUTPUT_DIR}")

## Load and Inspect Datasets

In [ ]:
# Load training data
print("Loading training data...")
bioma_aoi = gpd.read_file(TRAINING_DATA)

print(f"\nTraining Data Summary:")
print(f"  Polygons in AOI: {len(bioma_aoi):,}")
print(f"  CRS: {bioma_aoi.crs}")
print(f"  Columns: {list(bioma_aoi.columns)}")

In [ ]:
bioma_aoi.head(2)

In [ ]:
f, ax = plt.subplots(figsize=(10, 10))
bioma_aoi.plot("Land_us", legend=True, legend_kwds={'bbox_to_anchor': (1.05, 1), 'loc': 'upper left'}, cmap="tab20", ax=ax)

In [ ]:
# Translate Portuguese → English class names
class_translation = {
    'Mosaico Agricultura e pastagem': 'Agriculture and Pasture Mosaic',
    'Formação Florestal': 'Forest Formation (Native Forest)',
    'Pastagem': 'Pasture',
    'Floresta Plantada': 'Planted Forest (Commercial)',
    'Campo Alagado': 'Flooded Field/Wetland',
    'Formação Savânica': 'Savanna Formation',
    'Lavoura Perene': 'Perennial Crop',
    'Rio, Lago, Oceano': 'River, Lake, Ocean',
    'Lavoura Temporária': 'Temporary Crop',
    'Area Urbanizada': 'Urbanized Area',
    'Afloramento Rochoso': 'Rocky Outcrop',
    'Outras Areas nao Vegetadas': 'Other Non-Vegetated Areas'
}

bioma_aoi['Land_use_EN'] = bioma_aoi['Land_us'].map(class_translation)

# Class distribution
class_counts = bioma_aoi['Land_use_EN'].value_counts()
print("\nClass Distribution in AOI:")
print(class_counts.head(5))

# Calculate areas
bioma_aoi = ops_in_meters(bioma_aoi, estimate_area=True)

class_areas = bioma_aoi.groupby('Land_use_EN')['area_km2'].sum().sort_values(ascending=False)
print("\nClass Areas (km²):")
print(class_areas.round(1).head(5))

In [ ]:
f, ax = plt.subplots(figsize=(10, 10))
bioma_aoi.plot("Land_use_EN", legend=True, legend_kwds={'bbox_to_anchor': (1.05, 1), 'loc': 'upper left'}, cmap="tab20", ax=ax)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
def autopct_format(pct):
    """Only show percentage if >= 2%"""
    return f'{pct:.1f}%' if pct >= 2 else ''

wedges1, texts1, autotexts1 = axes[0].pie(class_counts, autopct=autopct_format, startangle=90)
axes[0].set_title('Class Distribution (Count)')
axes[0].legend(wedges1, class_counts.index, loc='center left', bbox_to_anchor=(1, 0, 0.5, 1), fontsize=8)

wedges2, texts2, autotexts2 = axes[1].pie(class_areas, autopct=autopct_format, startangle=90)
axes[1].set_title('Class Areas (km²)')
axes[1].legend(wedges2, class_areas.index, loc='center left', bbox_to_anchor=(1, 0, 0.5, 1), fontsize=8)
plt.tight_layout()

In [ ]:
# Combine similar classes for walkthrough purposes
classes_to_update = {
    "Temporary Crop": "Crop",
    "Perennial Crop": "Crop",
    "Agriculture and Pasture Mosaic": "Crop",
    "Pasture": "Pasture",
    "Savanna Formation": "Pasture"
}
bioma_aoi['Land_use_EN'] = bioma_aoi['Land_use_EN'].replace(classes_to_update)

# Recalculate class distribution and areas
class_counts = bioma_aoi['Land_use_EN'].value_counts()
print("\nClass Distribution in AOI:")
print(class_counts.head(5))
class_areas = bioma_aoi.groupby('Land_use_EN')['area_km2'].sum().sort_values(ascending=False)
print("\nClass Areas (km²):")
print(class_areas.round(1).head(5))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

wedges1, texts1, autotexts1 = axes[0].pie(class_counts, autopct=autopct_format, startangle=90)
axes[0].set_title('Class Distribution (Count)')
axes[0].legend(wedges1, class_counts.index, loc='center left', bbox_to_anchor=(1, 0, 0.5, 1), fontsize=8)
wedges2, texts2, autotexts2 = axes[1].pie(class_areas, autopct=autopct_format, startangle=90)
axes[1].set_title('Class Areas (km²)')
axes[1].legend(wedges2, class_areas.index, loc='center left', bbox_to_anchor=(1, 0, 0.5, 1), fontsize=8)

plt.tight_layout()


### Addressing Class Imbalance

In [ ]:
# Load Planet basemap
print("Loading Planet basemap...")
with rasterio.open(BASEMAP_PATH) as src:
    basemap_meta = src.meta.copy()
    basemap_crs = src.crs
    basemap_transform = src.transform
    basemap_bounds = src.bounds
    basemap_shape = (src.height, src.width)

print(f"\nBasemap Summary:")
print(f"  CRS: {basemap_crs}")
print(f"  Shape: {basemap_shape} (H × W)")
print(f"  Bands: {basemap_meta['count']}")
print(f"  Resolution: {abs(basemap_transform[0]):.2f}m")

In [ ]:
# full snippet 30 seconds to run
with rasterio.open(BASEMAP_PATH) as src:
    basemap_blue = src.read(1)
    basemap_green = src.read(2)
    basemap_red = src.read(3)
    basemap_transform = src.transform

# create a visualization of the basemap
pct_max = 99.5
blue_norm = basemap_blue / np.percentile(basemap_blue, pct_max)
green_norm = basemap_green / np.percentile(basemap_green, pct_max)
red_norm = basemap_red / np.percentile(basemap_red, pct_max)
vis_stacked = np.clip(np.dstack((red_norm, green_norm, blue_norm)), 0, 1)

# 20 seconds
f, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(vis_stacked)
ax.axis("off")
ax.set_title("Planet June 2025 Basemap (RGB)");


In [ ]:
plt.close()

## Training Point Sampling

Using the BioMa shapefile, we can create balanced point samples to train the classification model.

In [ ]:
# Sample training points from polygon interiors
def sample_training_points(polygons_gdf, min_samples_per_class=50, buffer_inward=5):
    """
    Sample points from polygon interiors with stratified sampling.

    Args:
        polygons_gdf: GeoDataFrame with training polygons
        min_samples_per_class: Minimum points per class
        buffer_inward: Buffer inward (meters) to avoid edges

    Returns:
        GeoDataFrame with sampled points and class labels
    """
    points_list = []

    for class_name in polygons_gdf['Land_use_EN'].unique():
        class_polygons = polygons_gdf[polygons_gdf['Land_use_EN'] == class_name].copy()

        class_polygons = ops_in_meters(class_polygons, buffer=-buffer_inward)
        # Remove invalid geometries after buffering
        class_polygons = class_polygons[class_polygons.is_valid & ~class_polygons.is_empty]

        if len(class_polygons) == 0:
            print(f"  ⚠️  {class_name}: No valid geometries after buffering")
            continue

        # Calculate points per polygon
        n_polygons = len(class_polygons)
        target_points = max(min_samples_per_class, n_polygons * 2)
        points_per_polygon = max(1, target_points // n_polygons)

        # Sample points
        for idx, row in class_polygons.iterrows():
            geom = row['geometry']

            # Sample points within polygon
            sampled_points = []
            attempts = 0
            max_attempts = points_per_polygon * 100

            while len(sampled_points) < points_per_polygon and attempts < max_attempts:
                # Random point within bounds
                minx, miny, maxx, maxy = geom.bounds
                random_point = Point(
                    np.random.uniform(minx, maxx),
                    np.random.uniform(miny, maxy)
                )

                if geom.contains(random_point):
                    sampled_points.append({
                        'geometry': random_point,
                        'class': class_name
                    })

                attempts += 1

            points_list.extend(sampled_points)

        print(f"  {class_name}: {len([p for p in points_list if p['class'] == class_name])} points sampled")

    # Create GeoDataFrame
    points_gdf = gpd.GeoDataFrame(points_list, crs=polygons_gdf.crs)

    return points_gdf

print("Sampling training points...")
training_points = sample_training_points(bioma_aoi, min_samples_per_class=50, buffer_inward=5)

print(f"\nTotal points sampled: {len(training_points):,}")
print(f"Classes represented: {training_points['class'].nunique()}")

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(8, 8))
# add grid
ax.grid(True, which='both', color='lightgray', linewidth=0.5, alpha=0.7)
bioma_aoi.plot("Land_use_EN", ax=ax, alpha=0.5, legend=True,
    legend_kwds={'loc': 'lower right', 'fontsize': 8, 'markerscale': 0.5})
training_points.plot(ax=ax, markersize=0.1, color="k")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_points_map.png"), dpi=300)

In [ ]:
training_points.to_file(os.path.join(OUTPUT_DIR, "training_points.geojson"), driver="GeoJSON")

## Feature Extraction

Next, we will extract features at every point location to train the model.

In [ ]:
training_points = gpd.read_file(os.path.join(OUTPUT_DIR, "training_points.geojson"))
training_points.head(2)

In [ ]:
def extract_spectral_features(raster_path, points_gdf, mask_path=None):
    """
    Extract spectral band values and calculate indices for training points.

    Args:
        raster_path: Path to Planet basemap (4-band: Blue, Green, Red, NIR)
        points_gdf: GeoDataFrame with point geometries
        mask_path: Optional path to UDM2 mask

    Returns:
        DataFrame with features [Blue, Green, Red, NIR, NDVI, NDWI, SAVI, EVI]
    """
    features_list = []

    with rasterio.open(raster_path) as src:
        print(f"src.height: {src.height}, src.width: {src.width}")
        # Reproject points to match raster CRS (CRITICAL FIX)
        if points_gdf.crs != src.crs:
            print("Reprojecting points to match raster CRS...")
            points_reproj = points_gdf.to_crs(src.crs)
            print(f"CRS: {src.crs.to_epsg()}, Points CRS: {points_reproj.crs.to_epsg()}")
        else:
            points_reproj = points_gdf.copy()

        # Load UDM2 if provided
        udm2 = None
        if mask_path:
            with rasterio.open(mask_path) as udm_src:
                udm2 = udm_src.read(1)  # Read first band

        for idx, point in tqdm(points_reproj.iterrows(), total=len(points_reproj), desc="Extracting features"):
            # Get pixel coordinates
            x, y = point.geometry.x, point.geometry.y
            row, col = rowcol(src.transform, x, y)

            # Check if within raster bounds
            if row < 0 or row >= src.height or col < 0 or col >= src.width:
                print(f"  ⚠️  Point {idx} is out of raster bounds, skipping")
                continue

            # Check UDM2 mask (band 1 = clear pixels are 0)
            if udm2 is not None and udm2[row, col] != 1:
                print(f"  ⚠️  Point {idx} is cloudy/poor quality (UDM2={udm2[row, col]}), skipping")
                continue  # Skip cloudy/poor quality pixels

            # Extract band values (Blue, Green, Red, NIR)
            blue = src.read(1, window=Window(col, row, 1, 1))[0, 0]
            green = src.read(2, window=Window(col, row, 1, 1))[0, 0]
            red = src.read(3, window=Window(col, row, 1, 1))[0, 0]
            nir = src.read(4, window=Window(col, row, 1, 1))[0, 0]

            # Handle nodata
            if np.isnan([blue, green, red, nir]).any():
                continue

            # Calculate spectral indices
            with np.errstate(divide='ignore', invalid='ignore'):
                ndvi = (nir - red) / (nir + red)
                ndwi = (green - nir) / (green + nir)
                savi = ((nir - red) / (nir + red + 0.5)) * 1.5
                evi = 2.5 * ((nir - red) / (nir + 6*red - 7.5*blue + 1))

            # Replace inf/nan with 0
            ndvi = 0 if np.isnan(ndvi) or np.isinf(ndvi) else ndvi
            ndwi = 0 if np.isnan(ndwi) or np.isinf(ndwi) else ndwi
            savi = 0 if np.isnan(savi) or np.isinf(savi) else savi
            evi = 0 if np.isnan(evi) or np.isinf(evi) else evi

            features_list.append({
                'Blue': blue,
                'Green': green,
                'Red': red,
                'NIR': nir,
                'NDVI': ndvi,
                'NDWI': ndwi,
                'SAVI': savi,
                'EVI': evi,
                'class': point['class']
            })

    return pd.DataFrame(features_list)

print("Extracting spectral features...")
features_df = extract_spectral_features(BASEMAP_PATH, training_points, mask_path=UDM2_PATH)

print("\nFeature extraction complete:")
print(f"  Valid samples: {len(features_df):,}")
print(f"  Features: {[c for c in features_df.columns if c != 'class']}")
print("\nFeature statistics:")
print(features_df.describe())

In [ ]:
features_df

In [ ]:
# Create feature matrix and train/test split
feature_cols = ['Blue', 'Green', 'Red', 'NIR', 'NDVI', 'NDWI', 'SAVI', 'EVI']
X = features_df[feature_cols].values
y = features_df['class'].values

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain/Test Split:")
print(f"  Training samples: {len(X_train):,}")
print(f"  Test samples: {len(X_test):,}")
print(f"  Features: {len(feature_cols)}")
print(f"  Classes: {np.unique(y).shape[0]}")

# Verify stratification
train_class_dist = pd.Series(y_train).value_counts(normalize=True)
test_class_dist = pd.Series(y_test).value_counts(normalize=True)

print(f"\nClass distribution (train vs test):")
comparison = pd.DataFrame({
    'Train %': train_class_dist * 100,
    'Test %': test_class_dist * 100
})
print(comparison.round(1))

## Model Training & Class Balancing

Stratified sampling used for points addresses class imbalance at the input
dataset level (from largest to smallest class, ratios go from 1518:1 → 47:1).

Next, we can use balanced class weights to ensure the model pays attention to
minority classes in training.

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Create weight mapping for visualization
class_names = np.unique(y_train)
weight_df = pd.DataFrame({
    'Class': class_names,
    'Train_Samples': [np.sum(y_train == c) for c in class_names],
    'Weight': class_weights
}).sort_values('Weight', ascending=False)

print("Class Weights (Balanced):")
print(weight_df.to_string(index=False))
print(f"\nWeight range: {class_weights.min():.3f} - {class_weights.max():.3f}")
print("Interpretation: Minority classes receive higher weights during training")

In [ ]:
# Configure Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=200,           # Choose a sufficient number of trees for stability
    max_depth=None,             # Allow full tree growth (prevents underfitting)
    min_samples_split=10,       # Prevent overfitting on small classes
    min_samples_leaf=5,         # Prevents memorization of rare class outliers
    class_weight='balanced',    # Apply computed class weights
    random_state=42,            # Set random state for reproducibility
    n_jobs=-1,                  # Use all CPU cores available
    oob_score=True,             # Out-of-bag score for training validation
    verbose=0
)

In [ ]:
# Train Random Forest Model
print("Training Random Forest classifier...")
start_time = time.time()
rf_model.fit(X_train, y_train)

training_time = time.time() - start_time

print("\n✅ Training complete!")
print(f"  Training time: {training_time:.1f} seconds")
print(f"  OOB Score: {rf_model.oob_score_:.4f}")
print(f"  Number of classes: {len(rf_model.classes_)}")
print(f"  Feature names: {feature_cols}")

# Training performance summary
print("\nTraining Performance:")
print(f"  Model has learned from {len(X_train):,} samples")
print(f"  OOB accuracy: {rf_model.oob_score_*100:.2f}%")
print(f"  Ready for evaluation on {len(X_test):,} test samples")

In [ ]:
# Feature Importance Analysis
feature_importance = rf_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 4))
importance_df.plot(kind='barh', x='Feature', y='Importance', ax=ax, color='forestgreen', legend=False)
ax.set_xlabel('Importance Score (0-1)')
ax.set_title('Random Forest Feature Importance')
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"  Most important feature: {importance_df.iloc[0]['Feature']} ({importance_df.iloc[0]['Importance']:.3f})")
print(f"  Least important feature: {importance_df.iloc[-1]['Feature']} ({importance_df.iloc[-1]['Importance']:.3f})")

# Check for redundant features
redundant_threshold = 0.05
redundant_features = importance_df[importance_df['Importance'] < redundant_threshold]
if len(redundant_features) > 0:
    print(f"\n⚠️  Features with <5% importance (potentially redundant):")
    print(redundant_features.to_string(index=False))
else:
    print(f"\n✅ All features contribute >5% importance")

In [ ]:
### Decision tree visualization
# Select a representative tree (e.g., tree 0 or a medium-depth tree)
tree_idx = 0  # Change this to visualize different trees (0-199)

# Find a tree with moderate depth for better visualization
tree_depths = [tree.get_depth() for tree in rf_model.estimators_]
median_depth = np.median(tree_depths)
moderate_trees = [i for i, depth in enumerate(tree_depths) if abs(depth - median_depth) <= 2]
tree_idx = moderate_trees[0] if moderate_trees else 0

# Create figure with appropriate size
fig, ax = plt.subplots(figsize=(25, 10))

# Plot the decision tree
tree_plot = plot_tree(
    rf_model.estimators_[tree_idx],
    feature_names=feature_cols,
    class_names=rf_model.classes_,
    filled=False,           # Color nodes by majority class
    rounded=True,          # Rounded box style
    fontsize=14,
    ax=ax,
    max_depth=4,          # Limit depth for readability (full tree can be overwhelming)
    impurity=False,        # Show gini impurity
    proportion=False,       # Show sample proportions instead of counts
)

for text in ax.texts:
    if 'value' in text.get_text():
        text_split = text.get_text().split('\n')
        node_text = [line if '<' in line or '>' in line else "" for line in text_split]
        if np.all([line == "" for line in node_text]):
            node_text = ["NODE END"]
        samples_text = [line.replace("samples", "N") if 'samples' in line else "" for line in text_split]
        replace_text = "\n".join([line.strip() for line in (node_text + samples_text) if line.strip()])
        text.set_text(replace_text)  # Keep only the class label


ax.set_title(f'Decision Tree {tree_idx} from Random Forest (max_depth=4 shown)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Model Evaluation & Validation

In this section, we will test prediction and accuracy metrics against the witheld labeled data.

In [ ]:
# Predict on test set
y_pred = rf_model.predict(X_test)

# Calculate overall accuracy
test_accuracy = accuracy_score(y_test, y_pred)

print("="*60)
print("TEST SET EVALUATION")
print("="*60)
print(f"\nOverall Test Accuracy: {test_accuracy*100:.2f}%")
print(f"OOB Training Accuracy: {rf_model.oob_score_*100:.2f}%")
print(f"Generalization Gap: {(rf_model.oob_score_ - test_accuracy)*100:.2f}%")

# Per-class performance metrics
print("\n" + "="*60)
print("PER-CLASS PERFORMANCE METRICS")
print("="*60)
class_report = classification_report(y_test, y_pred, target_names=rf_model.classes_, output_dict=True)
class_report_df = pd.DataFrame(class_report).transpose()

# Display key metrics
display_cols = ['precision', 'recall', 'f1-score', 'support']
print("\n" + class_report_df[display_cols].to_string())

# Summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"Macro Average F1: {class_report['macro avg']['f1-score']:.3f}")
print(f"Weighted Average F1: {class_report['weighted avg']['f1-score']:.3f}")
print(f"Classes with F1 > 0.7: {sum([1 for c in rf_model.classes_ if class_report[c]['f1-score'] > 0.7])}/{len(rf_model.classes_)}")
print(f"Classes with F1 < 0.5: {sum([1 for c in rf_model.classes_ if class_report[c]['f1-score'] < 0.5])}/{len(rf_model.classes_)}")

In [ ]:
# Confusion Matrix Analysis

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=rf_model.classes_)

# Calculate normalized confusion matrix (percentages)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Visualize confusion matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Absolute counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=rf_model.classes_,
            yticklabels=rf_model.classes_,
            ax=ax1, cbar_kws={'label': 'Count'})
ax1.set_xlabel('Predicted Class')
ax1.set_ylabel('True Class')
ax1.set_title('Confusion Matrix (Counts)')
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right', fontsize=9)
plt.setp(ax1.get_yticklabels(), rotation=0, fontsize=9)

# Normalized percentages
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=rf_model.classes_,
            yticklabels=rf_model.classes_,
            ax=ax2, vmin=0, vmax=1, cbar_kws={'label': 'Accuracy'})
ax2.set_xlabel('Predicted Class')
ax2.set_ylabel('True Class')
ax2.set_title('Confusion Matrix (Normalized)')
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=9)
plt.setp(ax2.get_yticklabels(), rotation=0, fontsize=9)

plt.tight_layout()
plt.show()

## Export the Model

In [ ]:
joblib.dump(rf_model, os.path.join(OUTPUT_DIR, "rf_model.joblib"))

In [ ]:
# Save model metadata
metadata = {
    'model_version': '1.0',
    'creation_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model_type': 'RandomForestClassifier',

    # Feature information
    'feature_names': ['Blue', 'Green', 'Red', 'NIR', 'NDVI', 'NDWI', 'SAVI', 'EVI'],
    'n_features': 8,
    'x_train_names': feature_cols,
    'x_test_names': feature_cols,
    'y_variable_name': 'class',

    # Class information
    'class_names': list(np.unique(y_train)),
    'n_classes': len(np.unique(y_train)),

    # Model parameters
    'model_params': {
        'n_estimators': rf_model.n_estimators,
        'max_depth': rf_model.max_depth,
        'min_samples_split': rf_model.min_samples_split,
        'min_samples_leaf': rf_model.min_samples_leaf,
        'class_weight': 'balanced',
        'random_state': rf_model.random_state,
        'n_jobs': rf_model.n_jobs,
        'oob_score': rf_model.oob_score
    },

    # Training information
    'training_info': {
        'n_training_samples': X_train.shape[0],
        'n_test_samples': X_test.shape[0],
        'oob_accuracy': rf_model.oob_score_,
        'training_date': datetime.now().strftime('%Y-%m-%d'),
        'basemap_date': '2025-06',  # June 2025
        'season': 'dry',
        'crs': 'EPSG:3857'  # Web Mercator (basemap CRS)
    }
}

# Save metadata as JSON
with open(os.path.join(OUTPUT_DIR, "rf_model_metadata.json"), 'w') as f:
    json.dump(metadata, f, indent=2)